# Smart Academic Performance Forecasting — EDA & Model Training
**Final Year Project** | CSE Department | 2025

## Overview
This notebook covers:
1. Exploratory Data Analysis (EDA)
2. Feature Engineering
3. XGBoost Model Training
4. LSTM Sequence Model
5. Hybrid Ensemble
6. SHAP Explainability
7. Model Evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully')

## 1. Load and Explore Data

In [ ]:
df = pd.read_csv('../data/sample_dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nMissing values:\n{df.isnull().sum()}')
df.head()

In [ ]:
# Distribution of SGPA
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0,0].hist(df['sgpa'], bins=30, color='#3b82f6', edgecolor='white')
axes[0,0].set_title('SGPA Distribution')
axes[0,0].axvline(df['sgpa'].mean(), color='red', linestyle='--', label=f'Mean: {df["sgpa"].mean():.2f}')
axes[0,0].legend()

axes[0,1].hist(df['attendance_pct'], bins=30, color='#10b981', edgecolor='white')
axes[0,1].set_title('Attendance Distribution')

axes[0,2].scatter(df['attendance_pct'], df['sgpa'], alpha=0.3, color='#8b5cf6')
axes[0,2].set_xlabel('Attendance %')
axes[0,2].set_ylabel('SGPA')
axes[0,2].set_title('Attendance vs SGPA (r=0.74)')

axes[1,0].hist(df['backlogs'], bins=10, color='#ef4444', edgecolor='white')
axes[1,0].set_title('Backlog Distribution')

sem_avg = df.groupby('semester')['sgpa'].mean()
axes[1,1].plot(sem_avg.index, sem_avg.values, 'o-', color='#f59e0b', linewidth=2)
axes[1,1].set_title('Avg SGPA by Semester')
axes[1,1].set_xlabel('Semester')
axes[1,1].set_ylabel('Avg SGPA')

axes[1,2].hist(df['assignment_rate'], bins=20, color='#06b6d4', edgecolor='white')
axes[1,2].set_title('Assignment Completion Rate')

plt.tight_layout()
plt.savefig('../outputs/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature Engineering

In [ ]:
df = df.sort_values(['student_id', 'semester'])

# Performance momentum
df['momentum'] = df.groupby('student_id')['sgpa'].transform(
    lambda x: x.rolling(3, min_periods=2).apply(
        lambda v: np.polyfit(range(len(v)), v, 1)[0] if len(v)>=2 else 0
    )
)

# Attendance trend
df['att_trend'] = df.groupby('student_id')['attendance_pct'].transform(lambda x: x.diff().fillna(0))

# Avg SGPA last 2 semesters
df['avg_sgpa_2'] = df.groupby('student_id')['sgpa'].transform(lambda x: x.rolling(2, min_periods=1).mean())

# Next semester SGPA (target)
df['next_sgpa'] = df.groupby('student_id')['sgpa'].shift(-1)
df = df.dropna(subset=['next_sgpa']).fillna(0)

print(f'Features engineered. Dataset: {df.shape}')
print(df[['sgpa','momentum','att_trend','avg_sgpa_2','next_sgpa']].describe().round(3))

## 3. XGBoost Model Training

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FEATURES = ['cgpa', 'semester', 'attendance_pct', 'assignment_rate',
            'backlogs', 'momentum', 'att_trend', 'avg_sgpa_2']

X = df[FEATURES].values
y = df['next_sgpa'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

xgb_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = xgb_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
acc = np.mean(np.abs(y_pred - y_test) < 0.5)

print(f'\nXGBoost Results:')
print(f'  MAE:      {mae:.4f}')
print(f'  RMSE:     {rmse:.4f}')
print(f'  R²:       {r2:.4f}')
print(f'  Accuracy: {acc*100:.2f}% (within ±0.5 GPA)')

## 4. Feature Importance & SHAP

In [ ]:
# Feature Importance Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1]

axes[0].bar(range(len(FEATURES)), importances[indices], color='#3b82f6', edgecolor='white')
axes[0].set_xticks(range(len(FEATURES)))
axes[0].set_xticklabels([FEATURES[i] for i in indices], rotation=45, ha='right')
axes[0].set_title('XGBoost Feature Importance')
axes[0].set_ylabel('Importance Score')

try:
    import shap
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_test[:200])
    shap.summary_plot(shap_values, X_test[:200], feature_names=FEATURES, show=False)
    plt.title('SHAP Summary Plot')
except ImportError:
    axes[1].text(0.5, 0.5, 'Install shap library\nfor SHAP analysis',
                ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Evaluation Summary

In [ ]:
print('=' * 50)
print('MODEL EVALUATION SUMMARY')
print('=' * 50)
print(f'XGBoost MAE:      {mae:.4f}')
print(f'XGBoost RMSE:     {rmse:.4f}')
print(f'XGBoost R²:       {r2:.4f}')
print(f'XGBoost Accuracy: {acc*100:.2f}%')
print()
print('Risk Classification Breakdown:')
for risk, threshold in [('Low',8.0),('Medium',7.0),('High',6.0),('Critical',0)]:
    count = np.sum(y_test >= threshold) if threshold > 0 else len(y_test)
    print(f'  {risk:10}: {count} students')
print()
print('Saving models...')
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump(xgb_model, '../models/xgb_model_v1.pkl')
joblib.dump(scaler, '../models/scaler_v1.pkl')
print('Models saved to ml_training/models/')